# Client Testing Notebook
Test each external client independently: Census Geocoder, US Congress API, Cicero, Google Civic, Tavily, and Anthropic.

In [1]:
import os
import json
import httpx
from dotenv import load_dotenv

load_dotenv("../.env")

print("GOOGLE_CIVIC_API_KEY:", "set" if os.environ.get("GOOGLE_CIVIC_API_KEY") else "MISSING")
print("TAVILY_API_KEY:", "set" if os.environ.get("TAVILY_API_KEY") else "MISSING")
print("ANTHROPIC_API_KEY:", "set" if os.environ.get("ANTHROPIC_API_KEY") else "MISSING")
print("CICERO_API_KEY:", "set" if os.environ.get("CICERO_API_KEY") else "MISSING")
print("US_CONGRESS_API_KEY:", "set" if os.environ.get("US_CONGRESS_API_KEY") else "MISSING")

GOOGLE_CIVIC_API_KEY: set
TAVILY_API_KEY: set
ANTHROPIC_API_KEY: set
CICERO_API_KEY: set
US_CONGRESS_API_KEY: set


## On the Issues (via local API)
Requires the backend running on `:8000` (`uvicorn main:app --reload`) and Cloud SQL proxy for the issues taxonomy.

In [2]:
BASE_URL = "http://localhost:8000"

# Step 1: Match a user query to an issue
resp_match = httpx.post(f"{BASE_URL}/api/issue-match", json={"query": "gun control"}, timeout=30)
print(f"Status: {resp_match.status_code}")
match_data = resp_match.json()
print(json.dumps(match_data, indent=2))

assert match_data["matched"], "Expected a match — is the issues taxonomy seeded?"
issue_id = match_data["issue"]["id"]
issue_label = match_data["issue"]["label"]
print(f"\nMatched: {issue_label} ({issue_id})")

Status: 200
{
  "matched": true,
  "issue": {
    "id": "gun_control",
    "label": "Gun Control"
  },
  "novel": false,
  "message": null
}

Matched: Gun Control (gun_control)


In [3]:
import time

# Step 2: Start issue research for a rep
resp_research = httpx.post(f"{BASE_URL}/api/issue-research", json={
    "representative": {"name": "Bernie Sanders", "office": "U.S. Senator", "level": "federal"},
    "issue_id": issue_id,
    "issue_label": issue_label,
}, timeout=30)
print(f"Status: {resp_research.status_code}")
research_data = resp_research.json()
print(json.dumps(research_data, indent=2))

research_id = research_data["research_id"]
status = research_data["status"]

# If cached, we're done. Otherwise poll.
if status != "complete":
    print(f"\nPolling research_id={research_id}...")
    for _ in range(30):
        time.sleep(2)
        poll = httpx.get(f"{BASE_URL}/api/issue-research/{research_id}", timeout=10).json()
        status = poll["status"]
        print(f"  status: {status}")
        if status in ("complete", "failed"):
            research_data = poll
            break

print(f"\nFinal status: {research_data['status']}")
print(json.dumps(research_data.get("summary"), indent=2))

Status: 200
{
  "research_id": "d45c59f7ef34",
  "status": "pending",
  "summary": null
}

Polling research_id=d45c59f7ef34...
  status: pending
  status: pending
  status: pending
  status: pending
  status: pending
  status: pending
  status: pending
  status: pending
  status: pending
  status: pending
  status: complete

Final status: complete
{
  "stance_summary": [
    "Sanders voted for the Bipartisan Safer Communities Act (S. 2938) in June 2022, which expanded background checks, enhanced mental health resources, and closed the so-called 'boyfriend loophole' in federal firearms law [1]. This was signed into law as Public Law 117-159 \u2014 the most significant federal gun legislation passed in decades [2].",
    "Sanders co-sponsored S. 3407, the Gun Violence Prevention and Community Safety Act of 2023, introduced by Sen. Elizabeth Warren, which proposed requiring firearm licensing, classifying semiautomatic assault weapons under the National Firearms Act, and other expanded gun